<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session07_document_processing_for_rag_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۷: آماده‌سازی اسناد فارسی برای RAG

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session07_document_processing_for_rag.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
!pip -q install pymupdf pypdf python-docx hazm sentence-transformers
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

import json
import re

import numpy as np
import pandas as pd
import pymupdf
from hazm import Normalizer, sent_tokenize
from hazm.utils import words_list
from llm_utils import download_data, load_store_docs
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

docx_path, pdf_path = download_data("novinkala_handbook.docx", "novinkala_handbook.pdf")

LEXICON = {w[0] if isinstance(w, tuple) else w for w in words_list()}
SUFFIXES = ("هایی", "های", "ها", "ای", "یی", "ی", "ش", "م")
WORD = re.compile(r"[\u0600-\u06FF\u200c]+")


def in_lexicon(word):
    word = word.replace("\u200c", "")
    return word in LEXICON or any(word.endswith(s) and word[:-len(s)] in LEXICON for s in SUFFIXES)


def lexicon_coverage(text):
    words = WORD.findall(text)
    return sum(in_lexicon(w) for w in words) / max(len(words), 1)


def fix_lam_alef(word):
    if "\u200c" in word:
        return "\u200c".join(fix_lam_alef(part) for part in word.split("\u200c"))
    if "ال" not in word or in_lexicon(word):
        return word
    for i in range(len(word) - 1):
        if word[i:i + 2] == "ال" and in_lexicon(word[:i] + "لا" + word[i + 2:]):
            return word[:i] + "لا" + word[i + 2:]
    return word


def clean_pdf_text(text):
    lines = []
    for line in text.split("\n"):
        line = line.strip()
        if line.startswith("."):
            line = line[1:].strip() + "."
        lines.append(line)
    text = re.sub(r"(?<![.:!؟?])\n(?!\n)", " ", "\n".join(lines))
    return WORD.sub(lambda m: fix_lam_alef(m.group()), text)


_normalizer = Normalizer(persian_numbers=False)
_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")


def normalize(text):
    return _normalizer.normalize(text).translate(_DIGITS)


def chunk_fixed(text, size=400, overlap=0):
    return [text[i:i + size] for i in range(0, len(text), max(size - overlap, 1))]


SEPARATORS = ["\n## ", "\n\n", "\n", ". ", "، ", " "]


def chunk_recursive(text, size=400, separators=SEPARATORS):
    if len(text) <= size:
        return [text.strip()] if text.strip() else []
    sep = next((s for s in separators if s in text), None)
    if sep is None:
        return chunk_fixed(text, size)
    parts = text.split(sep)
    pieces = [parts[0]] + [sep + p for p in parts[1:]]
    chunks, current = [], ""
    for piece in pieces:
        if len(current) + len(piece) <= size:
            current += piece
            continue
        if current.strip():
            chunks.append(current.strip())
        if len(piece) > size:
            chunks.extend(chunk_recursive(piece, size, separators[separators.index(sep) + 1:]))
            current = ""
        else:
            current = piece
    if current.strip():
        chunks.append(current.strip())
    return chunks


def chunk_by_sections(text, source, size=600):
    title = text.split("\n", 1)[0].lstrip("# ").strip()
    chunks = []
    for section in re.split(r"\n(?=## )", text):
        lines = section.strip().split("\n", 1)
        if lines[0].startswith("## "):
            heading, body = lines[0][3:].strip(), (lines[1] if len(lines) > 1 else "")
        else:
            heading, body = "", section.replace(lines[0], "", 1) if lines[0].startswith("# ") else section
        for piece in chunk_recursive(body.strip(), size):
            header = f"{title} > {heading}" if heading else title
            chunks.append({"source": source, "text": f"{header}\n{piece}"})
    return chunks


documents = {name: normalize(text) for name, text in load_store_docs().items()}
embedder = SentenceTransformer("intfloat/multilingual-e5-base")
questions = [json.loads(line) for line in open(download_data("rag_eval.jsonl"), encoding="utf-8")]
query_vectors = embedder.encode([f"query: {normalize(q['question'])}" for q in questions], normalize_embeddings=True)


def evaluate_chunks(name, chunks):
    vectors = embedder.encode([f"passage: {c['text']}" for c in chunks], normalize_embeddings=True)
    top3 = np.argsort(-(query_vectors @ vectors.T), axis=1)[:, :3]
    hits = [any(chunks[j]["source"] == q["source"] for j in top) for q, top in zip(questions, top3)]
    lengths = [len(embedder.tokenizer(c["text"]).input_ids) for c in chunks]
    return {"strategy": name, "chunks": len(chunks), "mean_tokens": round(np.mean(lengths), 1),
            "max_tokens": max(lengths), "hit_rate@3": round(np.mean(hits), 3)}

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

تابع `clean_pdf_text` را روی خروجی pypdf اجرا کنید. پوشش واژه‌نامه چقدر بهتر می‌شود؟ چرا اصلاح خروجی pypdf سخت‌تر است؟

</div>

In [ ]:
pypdf_text = "\n".join(page.extract_text() for page in PdfReader(pdf_path).pages)
mupdf_text = "\n".join(page.get_text() for page in pymupdf.open(pdf_path))

for name, text in [("pypdf", pypdf_text), ("pypdf + fixes", clean_pdf_text(pypdf_text)),
                   ("PyMuPDF", mupdf_text), ("PyMuPDF + fixes", clean_pdf_text(mupdf_text))]:
    print(f"{name:16} lexicon coverage: {lexicon_coverage(text):.1%}")

print("\n" + clean_pdf_text(pypdf_text)[:300])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

اصلاحات ما روی خروجی pypdf تقریباً هیچ اثری ندارند. نگاهی به متن علت را نشان می‌دهد: pypdf دو خرابی هم‌زمان ایجاد کرده است.

1. **ترتیب کلمات** هر خط برعکس شده، چون کلمه‌ها از چپ به راست کنار هم گذاشته شده‌اند.
2. **فاصله‌ها به نیم‌فاصله تبدیل شده‌اند**، پس کلمه‌های یک خط به هم چسبیده‌اند و یک «کلمهٔ» بسیار بلند ساخته‌اند، مثل «خرید‌سبد‌کردن‌نهایی‌هنگام». چنین رشته‌ای در واژه‌نامه نیست و اصلاح «لا» هم کمکی نمی‌کند.

برای اصلاح خروجی pypdf باید ابتدا نیم‌فاصله‌های اشتباه را از درست تشخیص داد، که خودش دشوار است، چون «می‌شود» نیم‌فاصلهٔ درست دارد، و بعد ترتیب کلمه‌ها را برعکس کرد، بدون اینکه بخش‌های انگلیسی و عددی برعکس شوند. هزینهٔ این کار از عوض کردن کتابخانه خیلی بیشتر است.

نکتهٔ دیگر دربارهٔ معیار ما: **پوشش واژه‌نامه فقط درستی تک‌تک کلمه‌ها را می‌سنجد و به ترتیب آن‌ها کور است.** اگر کلمه‌ها درست ولی برعکس بودند، نمرهٔ خوبی می‌گرفتند. برای سنجش ترتیب، معیاری مثل «درصد دوکلمه‌ای‌های مشترک با متن مرجع» لازم است. PyMuPDF ترتیب منطقی را درست بازسازی می‌کند و انتخاب بهتری است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

در مقایسه روش‌ها، اندازه تکه را برای روش «sections + headers» به ۲۰۰، ۴۰۰ و ۸۰۰ تغییر دهید. Hit Rate@3 و میانگین توکن چگونه تغییر می‌کنند؟

</div>

In [ ]:
rows = []
for size in [200, 400, 800]:
    chunks = [c for doc, text in documents.items() for c in chunk_by_sections(text, doc, size=size)]
    rows.append(evaluate_chunks(f"sections + headers {size}", chunks))
pd.DataFrame(rows)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با کوچک شدن اندازه، تعداد تکه‌ها زیاد و طول هر تکه کم می‌شود. اثر روی Hit Rate@3 دو جهت دارد:

- **تکه‌های کوچک‌تر** دقیق‌ترند، ولی چون تعداد گزینه‌ها بیشتر است، پیدا کردن تکه‌ای از سند درست در ۳ نتیجهٔ اول سخت‌تر می‌شود.
- **تکه‌های بزرگ‌تر** زمینهٔ بیشتری دارند و Hit Rate در سطح سند معمولاً بالاتر است، ولی پرامپت RAG را حجیم‌تر می‌کنند.

اسناد نوین‌کالا کوتاه‌اند و بیشتر بخش‌هایشان از ۴۰۰ کاراکتر کمترند. پس احتمالاً بین ۴۰۰ و ۸۰۰ تفاوت زیادی نمی‌بینید. اندازهٔ مناسب به طول بخش‌های اسناد شما بستگی دارد و باید با دادهٔ خودتان سنجیده شود.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

روش چهارمی بسازید که متن را با `sent_tokenize` از hazm به جمله‌ها بشکند و جمله‌ها را تا رسیدن به ۴۰۰ کاراکتر کنار هم بگذارد. آن را به جدول مقایسه اضافه کنید.

</div>

In [ ]:
def chunk_sentences(text, source, size=400):
    title = text.split("\n", 1)[0].lstrip("# ").strip()
    body = "\n".join(line for line in text.split("\n") if not line.startswith("#"))
    chunks, current = [], ""
    for sentence in sent_tokenize(body):
        if current and len(current) + len(sentence) + 1 > size:
            chunks.append({"source": source, "text": f"{title}\n{current.strip()}"})
            current = ""
        current += " " + sentence
    if current.strip():
        chunks.append({"source": source, "text": f"{title}\n{current.strip()}"})
    return chunks


strategies = {
    "fixed 400": lambda d, t: [{"source": d, "text": c} for c in chunk_fixed(t, 400, 80)],
    "recursive 400": lambda d, t: [{"source": d, "text": c} for c in chunk_recursive(t, 400)],
    "sections + headers": lambda d, t: chunk_by_sections(t, d),
    "sentences 400": lambda d, t: chunk_sentences(t, d),
}
pd.DataFrame([evaluate_chunks(name, [c for d, t in documents.items() for c in fn(d, t)])
              for name, fn in strategies.items()])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

تقسیم مبتنی بر جمله هیچ جمله‌ای را از وسط نمی‌برد، ولی مرز بخش‌ها را نمی‌شناسد: ممکن است انتهای بخش «ارسال عادی» و ابتدای «ارسال سریع» در یک تکه بیایند. ما عنوان سند را به هر تکه اضافه کردیم، ولی عنوان بخش را نه. می‌توانید هر دو ایده را ترکیب کنید: ابتدا بر اساس بخش تقسیم کنید و بعد داخل هر بخش جمله‌ها را گروه‌بندی کنید.

`sent_tokenize` در hazm جمله‌ها را با علائم پایان جمله (`.` و `؟` و `!`) جدا می‌کند. در متن‌هایی که علائم نگارشی ندارند، مثل پیام‌های چت، این روش خوب کار نمی‌کند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

یک سند Word یا PDF واقعی، مثلاً آیین‌نامه آموزشی دانشگاه یا یک دفترچه راهنما، انتخاب کنید و خط لوله این جلسه را روی آن اجرا کنید. پوشش واژه‌نامه چقدر است؟

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

سند خود را بارگذاری کنید. در Colab با اجرای سلول زیر پنجرهٔ انتخاب فایل باز می‌شود. روی سیستم خودتان کافی است `path` را برابر مسیر فایل قرار دهید.

</div>

In [ ]:
from docx import Document

try:
    from google.colab import files
    path = next(iter(files.upload()))
except ImportError:
    path = "my_document.pdf"          # change to the path of your file


def read_any(path):
    if path.lower().endswith(".docx"):
        return "\n\n".join(p.text for p in Document(path).paragraphs if p.text.strip())
    if path.lower().endswith(".pdf"):
        return clean_pdf_text("\n".join(page.get_text() for page in pymupdf.open(path)))
    return open(path, encoding="utf-8").read()


raw = read_any(path)
text = normalize(raw)
chunks = chunk_recursive(text, 500)
lengths = [len(embedder.tokenizer(c).input_ids) for c in chunks]
print(f"characters: {len(text):,}   lexicon coverage: {lexicon_coverage(text):.1%}")
print(f"chunks: {len(chunks)}   mean tokens: {np.mean(lengths):.0f}   max tokens: {max(lengths)}")
print("\n--- sample chunk ---\n" + chunks[len(chunks) // 2])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

برای تفسیر پوشش واژه‌نامه، آن را با یک متن مرجع از همان حوزه مقایسه کنید. متن‌های تخصصی، مثلاً حقوقی یا پزشکی، اصطلاحاتی دارند که در واژه‌نامهٔ عمومی hazm نیستند، پس حتی متن سالم هم ممکن است پوشش ۷۵ تا ۸۵ درصد داشته باشد. افت شدید، مثلاً زیر ۶۰ درصد، معمولاً نشانهٔ خرابی استخراج است. علت‌های رایج: PDF اسکن‌شده که متن ندارد و OCR می‌خواهد، فونت‌های قدیمی فارسی با کدگذاری غیراستاندارد، و جدول‌هایی که ستون‌هایشان در هم ریخته‌اند.

اگر سند شما ساختار عنوان دارد، `read_docx` جلسه را به کار ببرید تا عنوان‌ها حفظ شوند و از `chunk_by_sections` استفاده کنید.

</div>